In [25]:
import numpy as np
import csv, copy

import sys 
sys.path.append("/home/byzhou/desktop/python_code/Self_adaptive decoder/") 

from utils_files.RL_closed_loop_utils import *
from utils_files.closed_loop_simulator_Shah import *
from matplotlib import pyplot as plt

import time
import os 
import random
from tqdm import tqdm
import matplotlib.font_manager as font_manager

DEVICE = 'cuda'
model_name = 'SNN_streaming'

model_weight_name = "./model_weights/" + "OPS_" + model_name + "_classification" + "_model_state_dict.pth" 
neurons_save_path = "./model_weights/" + "OPS_" + model_name + "_classification" + "_neurons.csv" 
model_weight_name_update = "./model_weights/" + "Update_OPS_" + model_name + "_classification" + "_model_state_dict.pth" 

random_seed = 1234 
torch.manual_seed(random_seed)
np.random.seed(random_seed)
random.seed(random_seed)
if torch.cuda.is_available():
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.cuda.manual_seed_all(random_seed)
    
#### Parameters setup
time_step = 0.01 # time step is 10ms
max_duration = int(3/time_step) # maximum duration of each trial is 3s
side_radius = 10 # the radius of the target area is 10cm
max_vel = 20*time_step # cm/s
accel_const = 0.3 # acceleration constant 
min_distance = 8 # minimum distance between the target and the center of the workspace
target_size = 2.5 # the size of the target is 2.5cm
time_to_target = 0.5/time_step # the minimum time to target within the target area is 0.5s


vel_scale = 3.77
vel_label_max = max_vel/time_step/vel_scale
perturb_ratio = 0.6 # change the perturbation ratio 

neurons = 46 # the number of input neurons for the SNN model
output_neurons = 8 # the number of output neurons for the SNN model (4 classes for x direction and 4 classes for y direction)

    

In [26]:
### Closed-loop Models
class SNNModelStreamingContinuous(nn.Module):
    def __init__(self, input_dim=46, layer1=65, layer2=40, output_dim=4*2,
                 batch_size=512, sampling_rate=0.01, drop_rate=0.0,
                 beta=0.5, mem_thresh=0.5, spike_grad=surrogate.atan(alpha=2), 
                 vel_scale=3.77, error=0, sparse_rate=0, gamma_banditron=0.00001, gamma_agrel=0.001): 
        super().__init__()

        self.input_dim = input_dim
        self.output_dim = output_dim
        self.layer1 = layer1
        self.layer2 = layer2
        self.drop_rate = drop_rate
        self.vel_scale = vel_scale

        self.batch_size = batch_size
        self.sampling_rate = sampling_rate
        self.bin_window_size = self.batch_size

        self.fc1 = nn.Linear(self.input_dim, self.layer1, bias=False)
        self.fc2 = nn.Linear(self.layer1, self.layer2, bias=False)
        self.fc3 = nn.Linear(self.layer2, self.output_dim, bias=False)
        self.dropout = nn.Dropout(self.drop_rate)

        self.beta = beta
        self.mem_thresh = mem_thresh
        self.spike_grad = spike_grad
        self.lif1 = snn.Leaky(beta=0.5, spike_grad=self.spike_grad, threshold=0.6, learn_beta=True,
                              learn_threshold=True, init_hidden=True)
        self.lif2 = snn.Leaky(beta=0.6, spike_grad=self.spike_grad, threshold=0.4, learn_beta=True,
                              learn_threshold=True, init_hidden=True)
        self.lif3 = snn.Leaky(beta=0.5, spike_grad=self.spike_grad, threshold=0.6, learn_beta=True,
                              learn_threshold=True, init_hidden=True)

        self.register_buffer("data_buffer", torch.zeros(1, input_dim).type(torch.float32), persistent=False)
        self.register_buffer("label_buffer", torch.zeros(1, output_dim).type(torch.float32), persistent=False)
        self.input_count = 0
        self.softmax = torch.nn.Softmax(dim=1)
        self.activation = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

        self.error = error
        self.sparse_rate = sparse_rate
        self.gamma_banditron = gamma_banditron
        self.gamma_agrel = gamma_agrel
        
        self.lr_default = 5e-4
        self.lr = self.lr_default
            
        self.update_count = 0
        self.average_reward = []
        self.reset_lr_time_thre = 3 
        
        self.update_active = False
        self.banditron_update = False
        self.AGREL_update = False

    def reset_mem(self):
        self.lif1.reset_hidden()
        self.lif2.reset_hidden()
        self.lif3.reset_hidden()
    
    def grad_arctanh(self, u, alpha=2):
        grad = (1/math.pi)*(1/(1+(math.pi*u*alpha/2)**2))
        return grad
    
    def reset_lr(self):
        self.lr = self.lr_default

    def single_forward(self, x, label_):

        cur1 = self.dropout(self.fc1(x))
        spk1 = self.lif1(cur1)

        cur2 = self.dropout(self.fc2(spk1))
        spk2 = self.lif2(cur2)

        cur3 = self.fc3(spk2)
        spk3 = self.lif3(cur3)
        
        pred = torch.cat((self.lif3.mem[:, :int(self.output_dim/2)].clone().t(), self.lif3.mem[:, int(self.output_dim/2):].clone().t()), dim=1)
        

        if self.update_active:
            if self.banditron_update:
                self.update_count += 1
                pred = self.update_layer_banditron(spk2, pred, label_)
            elif self.AGREL_update:
                self.update_count += 1
                pred = self.update_layer_AGREL(input_=x, fc2input_=spk1, fc3input_=spk2, pred_=pred, label_=label_)


        return pred

    def forward(self, x, label_):
        predictions = []
        seq_length = x.shape[0]
        for seq in range(seq_length):
            self.input_count += 1
            current_seq = x[seq, :]
            self.data_buffer = torch.cat((self.data_buffer, current_seq.unsqueeze(0)), dim=0)

            self.data_buffer = self.data_buffer[1:, :]

            spikes = self.data_buffer.clone()

            pred = self.single_forward(spikes, label_[seq, :].unsqueeze(0))
            
            predictions.append(pred)
        predictions = torch.stack(predictions, dim=0)
        return predictions

    
    def update_layer_banditron(self, fc3input_, pred_, label_):
        ################## Banditron #######################
        y_hat = torch.argmax(pred_, dim=0)
        
        out_label = torch.zeros_like(pred_)
        for i, item in enumerate(y_hat):
            out_label[item, i] = 1
            
        p = ((1-self.gamma_banditron)*out_label + self.gamma_banditron/(self.output_dim/2)).numpy()
        
        y_tilde = torch.zeros_like(y_hat)
        y_tilde_y = torch.zeros_like(y_hat)
        y_tilde_prob = pred_
        for i in range(label_.shape[1]):
            y_tilde[i] = np.random.choice(range(int(self.output_dim/2)), p=p[:, i])
            y_tilde_y[i] = 1 if y_tilde[i] == label_[:,i] else 0
            y_tilde_prob[y_tilde[i], i] = 1

        sparsify = np.random.choice([True,False],p=[self.sparse_rate,1-self.sparse_rate])
        
        if not sparsify:
            dU = torch.zeros_like(self.fc3.weight)
            for out_order in range(label_.shape[1]):
                for r in range(int(self.output_dim//2)):
        
                    y_tilde_r = 1 if y_tilde[out_order] == r else 0
                    y_hat_r = 1 if y_hat[out_order] == r else 0
                    factor = ((y_tilde_y[out_order]*y_tilde_r)/p[r, out_order] - y_hat_r)

                    if out_order > 0:
                        r_ = r+int(self.output_dim//2)
                    else:
                        r_ = r
                    dU[r_,:] = fc3input_[0, :] * factor
            
            self.fc3.weight.data = self.fc3.weight.data + dU*self.lr
            
            if torch.max(self.fc3.weight.data) > 128 or torch.min(self.fc3.weight.data) < -127:
                self.fc3.weight.data = self.fc3.weight.data * 0.1
            

            self.update_count += 1
            
        return y_tilde_prob

    def update_layer_AGREL(self, input_, fc2input_, fc3input_, pred_, label_):
        ################ AGREL #######################
        pred_flat = torch.flatten(pred_.permute(1, 0)).unsqueeze(1)
        y_hat = torch.argmax(pred_, dim=0)
        y_hat[1] = y_hat[1]+int(self.output_dim//2)
        
        
        out_label = torch.zeros((pred_flat.shape[0], 2), dtype=torch.float32, device=input_.device)
        for i, item in enumerate(y_hat):
            out_label[item, i] = 1
        explore = np.random.uniform() < self.gamma_agrel
        
        # Evaluative framework (refer to the paper to understand the mathematics)
        y_tilde = torch.zeros_like(y_hat, device=input_.device)
        if explore:
            y_tilde_prob = pred_
            select_dim = np.random.randint(low=0,high=2)
            y_tilde[select_dim] = np.random.randint(low=0,high=int(self.output_dim//2))  #between 0-4
            y_tilde_prob[y_tilde[select_dim], select_dim] = 1
                
        else:
            y_tilde = y_hat
            y_tilde[1] = y_tilde[1]-int(self.output_dim//2)
            y_tilde_prob = pred_


        sparsify = np.random.choice([True,False],p=[self.sparse_rate,1-self.sparse_rate])
        r = torch.zeros_like(y_hat, device=input_.device)
        delta = torch.zeros_like(y_hat, dtype=torch.float32, device=input_.device)
        if not sparsify:
            for out_order in range(label_.shape[1]):
                if y_tilde[out_order] == label_[:, out_order]:
                    r[out_order] = 1
                    self.average_reward.append(r)
                    if out_order == 1:
                        delta[out_order] = r[out_order] - out_label[y_tilde[out_order]+int(self.output_dim//2), out_order]
                    else:
                        delta[out_order] = r[out_order] - out_label[y_tilde[out_order], out_order]
                    
                else:
                    r[out_order] = 0
                    self.average_reward.append(r)
                    delta[out_order] = np.random.choice([-1 ,1 -  out_label[y_tilde[out_order], out_order].item()],p=[1-self.error,self.error])
            
            ### update 3rd layer
            fb3 = (out_label @ delta.unsqueeze(1)).t()
            dW = self.lr*fc3input_.t() @ fb3
            

            ### update 2nd layer
            g2k = fc3input_
            fb2 = (((out_label) @ delta.unsqueeze(1)).t() @ self.fc3.weight.data) 
            dV = self.lr*(fc2input_.t()) @ (g2k*fb2) 

            
            ### update 1st layer
            g1j = fc2input_
            fb1 = (g2k * fb2) @ self.fc2.weight.data
            dU = self.lr*input_.t() @ (g1j*fb1) 
            
            self.fc3.weight.data = self.fc3.weight.data + dW.t()
            self.fc2.weight.data = self.fc2.weight.data + dV.t()
            self.fc1.weight.data = self.fc1.weight.data + dU.t()
            
        return y_tilde_prob

In [27]:
#### Run the closed-loop trials
def run_trials(model, cls, ops,num_trials,pos_total,output_total,intend_total,successful_trials,target_acq_time, model_weight_save=False):
    step = 0
    lr_reset_count = 0
    for trial in tqdm(range(num_trials)):
        cls.start_trial()
        t = 0
        t_in_range = 0
        lr_reset_count += 1
        
        pos_trial = []
        output_trial = []
        intend_vels = []
        target_lst = []
        start_point_lst = []
        
        model.reset_mem()

            
        while t < max_duration and t_in_range < time_to_target:
            step += 1
            vels,accels = cls.get_velocity()
            pos_trial.append(torch.tensor(copy.deepcopy(cls.position)).unsqueeze(0))
            intend_vels.append(copy.deepcopy(vels))
            target_lst.append(torch.tensor(cls.target).unsqueeze(0))
            start_point_lst.append(torch.tensor([[0,0]]))


            spikes = ops.get_spikes(accels)
            vels = torch.tensor(vels)/time_step
    

            pred_vels = model.forward(torch.tensor(spikes, dtype=torch.float32).unsqueeze(0), magnitude_to_label_class4(vels.unsqueeze(0)/vel_scale, max_mag=vel_label_max, min_mag=-vel_label_max, label_number=int(output_neurons//2)))
            pred_vels = label_to_magnitude_class4(torch.argmax(pred_vels, dim=1), max_mag=vel_label_max, min_mag=-vel_label_max, label_number=int(output_neurons//2))  ## from label to velocity
            
            output_trial.append(pred_vels.squeeze())
            cls.update_pos(pred_vels.squeeze().detach().numpy()*vel_scale*time_step) # update position in simulator
            t,t_in_range = cls.get_times()


        if (t_in_range >= time_to_target):
            successful_trials.append(1.0)
        else:
            successful_trials.append(0.0)

        target_acq_time.append(t)
        print("Trials Number", trial, "Time to Target", t*time_step)
        
        if model_weight_save:
            torch.save(model.state_dict(), model_weight_name_update)

        pos_trial = np.array(pos_trial)
        output_trial = np.array(output_trial)
        intend_vels = np.array(intend_vels)
        intend_vels = intend_vels / time_step
        output_trial = output_trial*vel_scale
        
        
        pos_total.append(pos_trial)
        output_total.append(output_trial)
        intend_total.append(intend_vels)


        if target_acq_time[-1]*time_step >= model.reset_lr_time_thre:
            model.reset_lr()
            lr_reset_count = 0
        
    return pos_total,output_total,intend_total,successful_trials,target_acq_time

In [28]:
def get_moving_avg(target_acq_time,window_size):
    acq_time_avg = np.zeros((len(target_acq_time)-window_size))
    for t in range(len(target_acq_time)-window_size):
        acq_time_avg[t] = np.mean(target_acq_time[t:t+window_size])
    return acq_time_avg

In [29]:
###################################### Simulations loop for the closed-loop trials without continus learning
simulation_result_time_to_target = []
simulation_result_award = []

for sim_num in range(10):

    cls = CLS(side_radius=side_radius,min_distance=min_distance,max_velocity=max_vel,
        accel_const=accel_const,target_size=target_size)


    ops = OPS(neurons,time_step,upper_lmin=5,lower_lmax=40,upper_lmax=100,
            max_accel=accel_const*max_vel,zero_prob=0.5)

    model = SNNModelStreamingContinuous(input_dim=neurons)
    model.load_state_dict(torch.load(model_weight_name_update, map_location=torch.device(DEVICE)), strict=False)
    model.eval()
    model.reset_mem()
    ops.assign_neurons(neurons_save_path)
    
    
    ###################################### normal closed_loop
    num_trials = 50

    successful_trials = []
    target_acq_time = []
    pos_total = []
    output_total = []
    intend_total = []
    error_rate = []
    
    model.update_active=False
    stage = 1
    pos_total,output_total,intend_total,successful_trials,target_acq_time = run_trials(model, cls, ops, num_trials,pos_total,output_total,intend_total,successful_trials,target_acq_time)

    ####################################### loss of neurons

    neurons_remaining = list(np.arange(neurons))
    num_select = int(neurons*perturb_ratio) # 30
    neurons_select = np.random.choice(neurons_remaining, num_select, replace=False)

    ops.remove_neurons(neurons_select)

    num_trials = 100

    model.update_active=True
    stage = 2
    pos_total,output_total,intend_total,successful_trials,target_acq_time = run_trials(model, cls, ops,num_trials,pos_total,output_total,intend_total,successful_trials,target_acq_time)


    simulation_result_time_to_target.append([i*time_step for i in target_acq_time])
    simulation_result_award.append(successful_trials)


ave_rewards = np.mean(np.array(simulation_result_award), axis=0)
ave_time_to_target = np.mean(np.array(simulation_result_time_to_target), axis=0)
ave_time_to_target = get_moving_avg(ave_time_to_target,4)


save_results(model_name=model_name, rewards=ave_rewards, time_to_target=ave_time_to_target, path="./Closed_loop/Results/")


  2%|▏         | 1/50 [00:00<00:11,  4.30it/s]

Trials Number 0 Time to Target 0.86


  6%|▌         | 3/50 [00:00<00:08,  5.71it/s]

Trials Number 1 Time to Target 0.86
Trials Number 2 Time to Target 0.84


 10%|█         | 5/50 [00:00<00:06,  7.38it/s]

...

Trials Number 96 Time to Target 3.0
Trials Number 97 Time to Target 0.86


 99%|█████████▉| 99/100 [00:34<00:00,  3.31it/s]

Trials Number 98 Time to Target 3.0


100%|██████████| 100/100 [00:34<00:00,  2.88it/s]

Trials Number 99 Time to Target 3.0


In [30]:
###################################### Simulations loop for the closed-loop trials with continus learning (AGREL)
simulation_result_time_to_target = []
simulation_result_award = []

for sim_num in range(10):

    cls = CLS(side_radius=side_radius,min_distance=min_distance,max_velocity=max_vel,
        accel_const=accel_const,target_size=target_size)


    ops = OPS(neurons,time_step,upper_lmin=5,lower_lmax=40,upper_lmax=100,
            max_accel=accel_const*max_vel,zero_prob=0.5)

    model = SNNModelStreamingContinuous(input_dim=neurons)
    model.load_state_dict(torch.load(model_weight_name_update, map_location=torch.device(DEVICE)), strict=False)
    model.eval()
    model.reset_mem()
    model.AGREL_update = True
    ops.assign_neurons(neurons_save_path)
    
    
    ###################################### normal closed_loop
    num_trials = 50

    successful_trials = []
    target_acq_time = []
    pos_total = []
    output_total = []
    intend_total = []
    error_rate = []
    
    model.update_active=False
    stage = 1
    pos_total,output_total,intend_total,successful_trials,target_acq_time = run_trials(model, cls, ops, num_trials,pos_total,output_total,intend_total,successful_trials,target_acq_time)

    ####################################### loss of neurons

    neurons_remaining = list(np.arange(neurons))
    num_select = int(neurons*perturb_ratio) # 30
    neurons_select = np.random.choice(neurons_remaining, num_select, replace=False)

    ops.remove_neurons(neurons_select)

    num_trials = 100

    model.update_active=True
    stage = 2
    pos_total,output_total,intend_total,successful_trials,target_acq_time = run_trials(model, cls, ops,num_trials,pos_total,output_total,intend_total,successful_trials,target_acq_time)


    simulation_result_time_to_target.append([i*time_step for i in target_acq_time])
    simulation_result_award.append(successful_trials)


ave_rewards = np.mean(np.array(simulation_result_award), axis=0)
ave_time_to_target = np.mean(np.array(simulation_result_time_to_target), axis=0)
ave_time_to_target = get_moving_avg(ave_time_to_target,4)


save_results(model_name=model_name+"_AGREL", rewards=ave_rewards, time_to_target=ave_time_to_target, path="./Closed_loop/Results/")


  2%|▏         | 1/50 [00:00<00:05,  9.57it/s]

Trials Number 0 Time to Target 0.79


  4%|▍         | 2/50 [00:00<00:05,  8.90it/s]

Trials Number 1 Time to Target 0.87


  6%|▌         | 3/50 [00:00<00:05,  8.61it/s]

...

Trials Number 95 Time to Target 0.91
Trials Number 96 Time to Target 0.86


 99%|█████████▉| 99/100 [00:14<00:00,  6.36it/s]

Trials Number 97 Time to Target 1.1500000000000001
Trials Number 98 Time to Target 0.9


100%|██████████| 100/100 [00:15<00:00,  6.61it/s]

Trials Number 99 Time to Target 1.3900000000000001


In [31]:
###################################### Simulations loop for the closed-loop trials with continus learning (Banditron)
simulation_result_time_to_target = []
simulation_result_award = []

for sim_num in range(10):

    cls = CLS(side_radius=side_radius,min_distance=min_distance,max_velocity=max_vel,
        accel_const=accel_const,target_size=target_size)


    ops = OPS(neurons,time_step,upper_lmin=5,lower_lmax=40,upper_lmax=100,
            max_accel=accel_const*max_vel,zero_prob=0.5)

    model = SNNModelStreamingContinuous(input_dim=neurons)
    model.load_state_dict(torch.load(model_weight_name_update, map_location=torch.device(DEVICE)), strict=False)
    model.eval()
    model.reset_mem()
    model.banditron_update = True
    ops.assign_neurons(neurons_save_path)
    
    
    ###################################### normal closed_loop
    num_trials = 50

    successful_trials = []
    target_acq_time = []
    pos_total = []
    output_total = []
    intend_total = []
    error_rate = []
    
    model.update_active=False
    stage = 1
    pos_total,output_total,intend_total,successful_trials,target_acq_time = run_trials(model, cls, ops, num_trials,pos_total,output_total,intend_total,successful_trials,target_acq_time)

    ####################################### loss of neurons

    neurons_remaining = list(np.arange(neurons))
    num_select = int(neurons*perturb_ratio) # 30
    neurons_select = np.random.choice(neurons_remaining, num_select, replace=False)

    ops.remove_neurons(neurons_select)

    num_trials = 100

    model.update_active=True
    stage = 2
    pos_total,output_total,intend_total,successful_trials,target_acq_time = run_trials(model, cls, ops,num_trials,pos_total,output_total,intend_total,successful_trials,target_acq_time)


    simulation_result_time_to_target.append([i*time_step for i in target_acq_time])
    simulation_result_award.append(successful_trials)


ave_rewards = np.mean(np.array(simulation_result_award), axis=0)
ave_time_to_target = np.mean(np.array(simulation_result_time_to_target), axis=0)
ave_time_to_target = get_moving_avg(ave_time_to_target,4)


save_results(model_name=model_name+"_Banditron", rewards=ave_rewards, time_to_target=ave_time_to_target, path="./Closed_loop/Results/")


  2%|▏         | 1/50 [00:00<00:06,  7.99it/s]

Trials Number 0 Time to Target 0.92


  4%|▍         | 2/50 [00:00<00:05,  8.38it/s]

Trials Number 1 Time to Target 0.86


  6%|▌         | 3/50 [00:00<00:05,  8.51it/s]

...

Trials Number 96 Time to Target 1.3
Trials Number 97 Time to Target 0.92


 99%|█████████▉| 99/100 [00:22<00:00,  5.43it/s]

Trials Number 98 Time to Target 0.93


100%|██████████| 100/100 [00:22<00:00,  4.48it/s]

Trials Number 99 Time to Target 1.58
